# Bronze → Silver: validate before publishing

The deployment replaces the first code cell with non-secret `CONFIG_JSON` parameters. This notebook runs in the provider's Fabric Spark runtime, not a local Python kernel. It reads only generated synthetic files, uses an explicit schema, rejects conflicting duplicate keys, validates firm isolation and foreign keys, then writes Delta tables. All validation happens before the first table write; Delta commits are atomic per table, not across the entire lakehouse. Failed partial runs must be rerun before sharing.

Validation status: offline notebook-format, Python AST, deployment-payload, and lint checks only. No Fabric execution has been performed; these checks do not establish Spark or Delta runtime success.


In [ ]:
# Deployment parameter cell; replaced in memory before upload.
CONFIG_JSON = '{}'


In [ ]:
import json
import re
from datetime import date
from time import perf_counter
from uuid import UUID

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.ansi.enabled', 'true')
config = json.loads(CONFIG_JSON)
required = {'source_workspace_id', 'source_lakehouse_id', 'target_workspace_id',
            'target_lakehouse_id', 'firm_slug', 'tables', 'schemas', 'as_of'}
if not required.issubset(config):
    raise ValueError('Run through fabric/deploy.py: complete notebook configuration is required')
for key in ('source_workspace_id', 'source_lakehouse_id', 'target_workspace_id', 'target_lakehouse_id'):
    UUID(config[key])
if config['source_workspace_id'] != config['target_workspace_id']:
    raise ValueError('Provider transformation must not execute across tenant workspaces')
firm = config['firm_slug']
if not re.fullmatch(r'[a-z]+', firm):
    raise ValueError('Invalid firm slug')
for table in config['tables']:
    if not re.fullmatch(r'[a-z_]+', table):
        raise ValueError('Invalid table name')
snapshot = date.fromisoformat(config['as_of'])
source = f"abfss://{config['source_workspace_id']}@onelake.dfs.fabric.microsoft.com/{config['source_lakehouse_id']}"
target = f"abfss://{config['target_workspace_id']}@onelake.dfs.fabric.microsoft.com/{config['target_lakehouse_id']}"
if source == target:
    raise ValueError('Source and target lakehouses must differ')
frames = {}
counts = {}
started = perf_counter()
for table in config['tables']:
    spec = config['schemas'][table]
    frame = spark.read.schema(StructType.fromJson(spec['spark_schema'])).parquet(
        f'{source}/Files/{firm}/{table}.parquet')
    # Remove exact duplicates only. Conflicting duplicate keys are rejected below.
    frame = frame.dropDuplicates().cache()
    key = spec['primary_key']
    if frame.groupBy(key).count().where('count > 1').limit(1).count():
        raise ValueError(f'Conflicting primary key: {table}')
    for field in spec['spark_schema']['fields']:
        if not field['nullable'] and frame.where(F.col(field['name']).isNull()).limit(1).count():
            raise ValueError(f"Required value missing: {table}.{field['name']}")
        if field['type'] == 'date':
            # Due dates and validity-window endpoints legitimately extend beyond snapshot.
            if field['name'] in {'work_date', 'incurred_on', 'issued_on', 'paid_on', 'adjusted_on', 'prepared_on'}:
                if frame.where(F.col(field['name']) > F.lit(snapshot)).limit(1).count():
                    raise ValueError(f'Future activity in {table}')
    if frame.where(F.col('firm_id') != F.lit(firm + '_f001')).limit(1).count():
        raise ValueError(f'Cross-firm row in {table}')
    frames[table] = frame
    counts[table] = frame.count()
for table, frame in frames.items():
    for fk in config['schemas'][table]['foreign_keys']:
        column = fk['column']
        # firm_id -> firms is already checked by the exact firm ID above.
        if column == 'firm_id':
            continue
        parent = frames[fk['table']].select(F.col(fk['target_column']).alias(column), 'firm_id')
        if frame.where(F.col(column).isNotNull()).join(parent, [column, 'firm_id'], 'left_anti').limit(1).count():
            raise ValueError(f'Orphan or cross-firm FK: {table}.{column}')
for table, frame in frames.items():
    location = f'{target}/Tables/{firm}_{table}'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(location)
    if spark.read.format('delta').load(location).count() != counts[table]:
        raise ValueError(f'Persisted row count mismatch: {table}')
    frame.unpersist()
print(json.dumps({'stage': 'bronze_to_silver', 'firm': firm, 'as_of': str(snapshot),
                  'rows': counts, 'elapsed_seconds': round(perf_counter() - started, 3)}))
